In [1]:
import os

notebook_dir = os.path.dirname(os.path.abspath("day05_agent_v2.ipynb"))
root_dir = os.path.dirname(notebook_dir)
os.chdir(root_dir)

print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\Subramani Mokkala\Desktop\arc-agi3-research


In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import arc_agi

load_dotenv()

arc = arc_agi.Arcade(arc_api_key = os.getenv("ARC_API_KEY"))
envs = arc.get_environments()

print("Connected, environments loaded:", len(envs))

INFO:arc_agi.scorecard:Initialized ScorecardManager with idle_for=0:15:00 and max_open_for=3 days, 0:00:00
2026-06-18 17:19:43 | INFO | Successfully fetched 25 environment(s) from API
Connected, environments loaded: 25


In [3]:
env = arc.make(game_id = "ls20", save_recording = False)
obs = env.reset()

game = env._game
print(f"Player position: {game.gudziatsk.x}, {game.gudziatsk.y}")
print(f"Player shape: {game.fwckfzsyc}")
print(f"Player color: {game.hiaauhahz}")
print(f"Player rotation: {game.cklxociuu}")
print(f"Collectibles: {game.plrpelhym}")

2026-06-18 17:20:16 | INFO | Created new scorecard: 8b6a20b0-13d5-4e31-bc7a-26f75e921e10
2026-06-18 17:20:18 | INFO | Successfully fetched metadata for game ls20
2026-06-18 17:20:18 | INFO | Successfully loaded game class Ls20 from environment_files\ls20\9607627b\ls20.py
Player position: 34, 45
Player shape: 5
Player color: 1
Player rotation: 3
Collectibles: [<arcengine.sprites.Sprite object at 0x000002419F569B20>]


In [5]:
for i, sprite in enumerate(game.plrpelhym):
    collectible_position = (sprite.x, sprite.y)
    required_shape = game.ldxlnycps[i]
    required_color = game.yjdexjsoa[i]
    required_rotation = game.ehwheiwsk[i]
    print(f"  collectible {i}: pos={collectible_position} requires shape={required_shape} color={required_color} rotation={required_rotation}")

  collectible 0: pos=(34, 10) requires shape=5 color=1 rotation=0


In [12]:
ACTION_UP = 1
ACTION_DOWN = 2
ACTION_LEFT = 3
ACTION_RIGHT = 4

In [4]:
def get_player_pos(game):
    return (game.gudziatsk.x, game.gudziatsk.y)

def get_player_state(game):
    return {
        "shape": game.fwckfzsyc,
        "color": game.hiaauhahz,
        "rotation": game.cklxociuu
    }

In [17]:
def get_next_pos(current, action):
    if action == ACTION_UP:
        return (current[0], current[1]-1)
    elif action == ACTION_DOWN:
        return (current[0], current[1]+1)
    elif action == ACTION_LEFT:
        return (current[0]-1, current[1])
    elif action == ACTION_RIGHT:
        return (current[0]+1, current[1])
    else:
        return current

In [ ]:
import random

from matplotlib.style import available

def choose_action(current, target, blocked, avoid = []):
    dx = target[0] - current[0]
    dy = target[1] - current[1]

    if abs(dx) > abs(dy):
        if dx < 0:
            next_pos = (current[0]-1, current[1])
            if ACTION_LEFT not in blocked and next_pos not in avoid:
                return ACTION_LEFT
        elif dx > 0:
            next_pos = (current[0]+1, current[1])
            if ACTION_RIGHT not in blocked and next_pos not in avoid:
                return ACTION_RIGHT
    else:
        if dy < 0:
            next_pos = (current[0], current[1]-1)
            if ACTION_UP not in blocked and next_pos not in avoid:
                return ACTION_UP
        elif dy > 0:
            next_pos = (current[0], current[1]+1)
            if ACTION_DOWN not in blocked and next_pos not in avoid:
                return ACTION_DOWN
    available = [a for a in [ACTION_UP, ACTION_DOWN, ACTION_LEFT, ACTION_RIGHT] 
             if a not in blocked and get_next_pos(current, a) not in avoid]
    if not available:
        available = [a for a in [ACTION_UP, ACTION_DOWN, ACTION_LEFT, ACTION_RIGHT] 
                 if a not in blocked]
    if not available:
        available = [ACTION_UP, ACTION_DOWN, ACTION_LEFT, ACTION_RIGHT]
    return random.choice(available)


In [18]:
def is_stuck(pos_before, pos_after):
    return pos_before == pos_after

def get_level_plan(game):
    player_pos = get_player_pos(game)
    player_state = get_player_state(game)

    for i, sprite in enumerate(game.plrpelhym):
        collectible_pos = (sprite.x, sprite.y)
        required_shape = game.ldxlnycps[i]
        required_color = game.yjdexjsoa[i]
        required_rotation = game.ehwheiwsk[i]

        if (player_state["rotation"] != required_rotation):
            return (19, 30)
        if (player_state["color"] != required_color):
            return (49, 45)
        if (player_state["shape"] != required_shape):
            return None
        return collectible_pos

In [10]:
obs = env.reset()
step = 0
max_steps = 500
blocked = set()
prev_pos = None
last_action = None

print("starting agent")

while obs.state.value == "NOT_FINISHED" and step < max_steps:
    
    current = get_player_pos(env._game)
    target = get_level_plan(env._game)
    
    if target is None:
        break
    
    if prev_pos is not None and is_stuck(prev_pos, current):
        blocked.add(last_action)
    else:
        blocked = set()
    
    action = choose_action(current, target, blocked)
    
    prev_pos = current
    last_action = action
    obs = env.step(action)
    step += 1
    
    if step % 20 == 0:
        print(f"step {step} | levels: {obs.levels_completed} | pos: {current}")

print(f"game ended after {step} steps")
print(f"final state: {obs.state.value}")
print(f"levels completed: {obs.levels_completed} / {obs.win_levels}")

starting agent
step 20 | levels: 0 | pos: (34, 30)
step 40 | levels: 0 | pos: (34, 40)
step 60 | levels: 0 | pos: (34, 30)
step 80 | levels: 0 | pos: (29, 25)
step 100 | levels: 1 | pos: (29, 35)
step 120 | levels: 1 | pos: (29, 35)
step 140 | levels: 1 | pos: (34, 35)
game ended after 149 steps
final state: GAME_OVER
levels completed: 1 / 7


In [13]:
obs = env.reset()

for i in range(105):
    current = get_player_pos(env._game)
    target = get_level_plan(env._game)
    if target is None:
        break
    if prev_pos is not None and is_stuck(prev_pos, current):
        blocked.add(last_action)
    else:
        blocked = set()
    action = choose_action(current, target, blocked)
    prev_pos = current
    last_action = action
    obs = env.step(action)

print(f"levels completed: {obs.levels_completed}")
print(f"player pos: {get_player_pos(env._game)}")
print(f"player state: {get_player_state(env._game)}")
for i, sprite in enumerate(game.plrpelhym):
    print(f"collectible {i}: pos=({sprite.x},{sprite.y}) requires shape={game.ldxlnycps[i]} color={game.yjdexjsoa[i]} rotation={game.ehwheiwsk[i]}")

levels completed: 0
player pos: (29, 35)
player state: {'shape': 5, 'color': 1, 'rotation': 0}
collectible 0: pos=(14,40) requires shape=5 color=1 rotation=3


In [14]:
obs = env.reset()
prev_pos = None
last_action = None
blocked = set()

print(f"player pos: {get_player_pos(env._game)}")
print(f"player state: {get_player_state(env._game)}")
for i, sprite in enumerate(game.plrpelhym):
    print(f"collectible {i}: pos=({sprite.x},{sprite.y}) requires shape={game.ldxlnycps[i]} color={game.yjdexjsoa[i]} rotation={game.ehwheiwsk[i]}")

player pos: (29, 40)
player state: {'shape': 5, 'color': 1, 'rotation': 0}
collectible 0: pos=(14,40) requires shape=5 color=1 rotation=3


In [15]:
env = arc.make(game_id="ls20", save_recording=False)
obs = env.reset()
game = env._game
prev_pos = None
last_action = None
blocked = set()

print(f"player pos: {get_player_pos(game)}")
print(f"player state: {get_player_state(game)}")
for i, sprite in enumerate(game.plrpelhym):
    print(f"collectible {i}: pos=({sprite.x},{sprite.y}) requires shape={game.ldxlnycps[i]} color={game.yjdexjsoa[i]} rotation={game.ehwheiwsk[i]}")

2026-06-16 17:58:41 | INFO | Successfully fetched metadata for game ls20
player pos: (34, 45)
player state: {'shape': 5, 'color': 1, 'rotation': 3}
collectible 0: pos=(34,10) requires shape=5 color=1 rotation=0


In [18]:
env = arc.make(game_id="ls20", save_recording=False)
game = env._game
obs = env.reset()

step = 0
max_steps = 500
blocked = set()
prev_pos = None
last_action = None

print("starting agent")

while obs.levels_completed < 1 and obs.state.value == "NOT_FINISHED" and step < max_steps:
    current = get_player_pos(game)
    target = get_level_plan(game)
    if target is None:
        break
    if prev_pos is not None and is_stuck(prev_pos, current):
        blocked.add(last_action)
    else:
        blocked = set()
    action = choose_action(current, target, blocked)
    prev_pos = current
    last_action = action
    obs = env.step(action)
    step += 1

print(f"Player pos: {get_player_pos(game)}")
print(f"Player state: {get_player_state(game)}")
for i, sprite in enumerate(game.plrpelhym):
    print(f"collectible {i}: pos=({sprite.x},{sprite.y}) requires shape={game.ldxlnycps[i]} color={game.yjdexjsoa[i]} rotation={game.ehwheiwsk[i]}")

2026-06-16 18:03:26 | INFO | Successfully fetched metadata for game ls20
starting agent
Player pos: (29, 40)
Player state: {'shape': 5, 'color': 1, 'rotation': 0}
collectible 0: pos=(14,40) requires shape=5 color=1 rotation=3


In [19]:
print(f"initial levels_completed: {obs.levels_completed}")
print(f"initial state: {obs.state.value}")

initial levels_completed: 1
initial state: NOT_FINISHED


In [20]:
for sprite in game.current_level._sprites:
    if sprite.tags:
        print(f"  pos=({sprite.x},{sprite.y}) tags={sprite.tags}")

  pos=(1,53) tags=['eqatonpohu']
  pos=(1,53) tags=['ghizzeqtoh']
  pos=(13,39) tags=['hoswmpiqkw']
  pos=(4,0) tags=['ihdgageizm']
  pos=(9,0) tags=['ihdgageizm']
  pos=(4,5) tags=['ihdgageizm']
  pos=(14,0) tags=['ihdgageizm']
  pos=(19,0) tags=['ihdgageizm']
  pos=(24,0) tags=['ihdgageizm']
  pos=(29,0) tags=['ihdgageizm']
  pos=(39,0) tags=['ihdgageizm']
  pos=(44,0) tags=['ihdgageizm']
  pos=(49,0) tags=['ihdgageizm']
  pos=(54,0) tags=['ihdgageizm']
  pos=(59,0) tags=['ihdgageizm']
  pos=(4,10) tags=['ihdgageizm']
  pos=(4,15) tags=['ihdgageizm']
  pos=(4,20) tags=['ihdgageizm']
  pos=(4,25) tags=['ihdgageizm']
  pos=(4,30) tags=['ihdgageizm']
  pos=(4,35) tags=['ihdgageizm']
  pos=(59,15) tags=['ihdgageizm']
  pos=(59,20) tags=['ihdgageizm']
  pos=(59,25) tags=['ihdgageizm']
  pos=(59,30) tags=['ihdgageizm']
  pos=(59,20) tags=['ihdgageizm']
  pos=(59,25) tags=['ihdgageizm']
  pos=(59,30) tags=['ihdgageizm']
  pos=(59,35) tags=['ihdgageizm']
  pos=(59,40) tags=['ihdgageizm']
  p

In [7]:
def find_sprite_by_tag(game, tag):
    for sprite in game.current_level._sprites:
        if tag in sprite.tags:
            return (sprite.x, sprite.y)
    return None

In [22]:
print(find_sprite_by_tag(game, "rhsxkxzdjz"))
print(find_sprite_by_tag(game, "rjlbuycveu"))
print(find_sprite_by_tag(game, "npxgalaybz"))

(49, 45)
(14, 40)
(15, 16)


In [19]:
def get_level_plan(game):
    player_pos = get_player_pos(game)
    player_state = get_player_state(game)

    for i, sprite in enumerate(game.plrpelhym):
        collectible_pos = (sprite.x, sprite.y)
        required_shape = game.ldxlnycps[i]
        required_color = game.yjdexjsoa[i]
        required_rotation = game.ehwheiwsk[i]

        if (player_state["rotation"] != required_rotation):
            return (find_sprite_by_tag(game, "rhsxkxzdjz"), [])
        if (player_state["color"] != required_color):
            return (find_sprite_by_tag(game, "soyhouuebz"), [])
        if (player_state["shape"] != required_shape):
            return (find_sprite_by_tag(game, "ttfwljgohq"), [])
        return (collectible_pos, [find_sprite_by_tag(game, "rhsxkxzdjz"), find_sprite_by_tag(game, "soyhouuebz"), find_sprite_by_tag(game, "ttfwljgohq")])

In [27]:
env = arc.make(game_id="ls20", save_recording=False)
game = env._game
obs = env.reset()
step = 0
max_steps = 500
blocked = set()
avoid = []
prev_pos = None
last_action = None

print("starting agent")

while obs.state.value == "NOT_FINISHED" and step < max_steps:
    current = get_player_pos(game)
    target, avoid = get_level_plan(game)
    if 30 <= step <= 55:
        print(f"step {step} | rotation: {get_player_state(game)['rotation']} | lives: {game.aqygnziho} | target: {target} | pos: {current}")
    if target is None:
        break
    if prev_pos is not None and is_stuck(prev_pos, current):
        blocked.add(last_action)
    else:
        blocked = set()
    action = choose_action(current, target, blocked, avoid)
    prev_pos = current
    last_action = action
    obs = env.step(action)
    step += 1

    if step % 20 == 0:
        print(f"step {step} | levels: {obs.levels_completed} | pos: {current} | target: {target}")

print(f"\ngame ended after {step} steps")
print(f"final state: {obs.state.value}")
print(f"levels completed: {obs.levels_completed} / {obs.win_levels}")

2026-06-18 18:04:21 | INFO | Successfully fetched metadata for game ls20
starting agent
step 20 | levels: 0 | pos: (34, 35) | target: (19, 30)
step 30 | rotation: 3 | lives: 3 | target: (19, 30) | pos: (29, 25)
step 31 | rotation: 3 | lives: 3 | target: (19, 30) | pos: (24, 25)
step 32 | rotation: 3 | lives: 3 | target: (19, 30) | pos: (24, 30)
step 33 | rotation: 0 | lives: 3 | target: (34, 10) | pos: (19, 30)
step 34 | rotation: 0 | lives: 3 | target: (34, 10) | pos: (19, 25)
step 35 | rotation: 0 | lives: 3 | target: (34, 10) | pos: (19, 25)
step 36 | rotation: 0 | lives: 3 | target: (34, 10) | pos: (14, 25)
step 37 | rotation: 0 | lives: 3 | target: (34, 10) | pos: (19, 25)
step 38 | rotation: 0 | lives: 3 | target: (34, 10) | pos: (19, 25)
step 39 | rotation: 0 | lives: 3 | target: (34, 10) | pos: (24, 25)
step 40 | levels: 0 | pos: (24, 25) | target: (34, 10)
step 40 | rotation: 0 | lives: 3 | target: (34, 10) | pos: (24, 25)
step 41 | rotation: 0 | lives: 3 | target: (34, 10) | 

In [21]:
env2 = arc.make(game_id="ls20", save_recording=False)
game2 = env2._game
obs2 = env2.reset()

print("all non-wall sprites in level 1:")
for sprite in game2.current_level._sprites:
    if sprite.tags and "ihdgageizm" not in sprite.tags:
        print(f"  pos=({sprite.x},{sprite.y}) tags={sprite.tags}")

2026-06-18 17:54:05 | INFO | Successfully fetched metadata for game ls20
all non-wall sprites in level 1:
  pos=(1,53) tags=['eqatonpohu']
  pos=(1,53) tags=['ghizzeqtoh']
  pos=(33,9) tags=['hoswmpiqkw']
  pos=(35,11) tags=['kvynsvxbpi']
  pos=(19,30) tags=['rhsxkxzdjz']
  pos=(34,10) tags=['rjlbuycveu']
  pos=(34,45) tags=['sfqyzhzkij']
  pos=(33,9) tags=['vjotnebuqo']
  pos=(3,55) tags=['wgmbtyhvbc']


In [23]:
print("moving sprites (gbvqrjtaqo):")
for sprite in game2.current_level._sprites:
    if sprite.tags and "gbvqrjtaqo" in sprite.tags:
        print(f"  pos=({sprite.x},{sprite.y}) tags={sprite.tags}")

print("\nhasivfwip (obstacle sprites):")
for obs_sprite in game2.hasivfwip:
    print(f"  type: {type(obs_sprite)}")
    print(f"  attrs: {[a for a in dir(obs_sprite) if not a.startswith('_')]}")

moving sprites (gbvqrjtaqo):

hasivfwip (obstacle sprites):


In [24]:
# check what happens to rotation when a life is lost
# run the agent until rotation changes from 0 to something else
env3 = arc.make(game_id="ls20", save_recording=False)
game3 = env3._game
obs3 = env3.reset()

print(f"lives: {game3.aqygnziho} | rotation: {game3.cklxociuu}")

# navigate to rotation changer
for _ in range(40):
    current = get_player_pos(game3)
    target = (19, 30)
    action = choose_action(current, target, set())
    obs3 = env3.step(action)
    
    if game3.cklxociuu == 0:
        print(f"rotation fixed to 0, lives: {game3.aqygnziho}")
        break

# now check lives before and after losing one
print(f"\nbefore life loss: lives={game3.aqygnziho} rotation={game3.cklxociuu} pos={get_player_pos(game3)}")

# press into a wall repeatedly to drain the step counter
for _ in range(100):
    obs3 = env3.step(ACTION_UP)
    if game3.aqygnziho < 3:
        print(f"life lost! lives={game3.aqygnziho} rotation={game3.cklxociuu} pos={get_player_pos(game3)}")
        break

2026-06-18 18:01:31 | INFO | Successfully fetched metadata for game ls20
lives: 3 | rotation: 3

before life loss: lives=3 rotation=3 pos=(34, 40)
life lost! lives=2 rotation=3 pos=(34, 45)


In [26]:
env3 = arc.make(game_id="ls20", save_recording=False)
game3 = env3._game
obs3 = env3.reset()

for i in range(15):
    current = get_player_pos(game3)
    target = (19, 30)
    action = choose_action(current, target, set())
    obs3 = env3.step(action)
    print(f"step {step} | rotation: {get_player_state(game)['rotation']} | lives: {game.aqygnziho} | target: {target} | pos: {current}")

2026-06-18 18:03:15 | INFO | Successfully fetched metadata for game ls20
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 45)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives: 0 | target: (19, 30) | pos: (34, 40)
step 131 | rotation: 3 | lives

In [29]:
with open("environment_files/ls20/9607627b/ls20.py", "r") as f:
    lines = f.readlines()

matches = [(i+1, line.strip()) for i, line in enumerate(lines) if "aqygnziho" in line]
for lineno, text in matches:
    print(f"line {lineno}: {text}")

line 1551: frame[toxqhfsju : toxqhfsju + 2, wmdkkxqkw + x] = tqogkgimes if self.vjhajnbdzr.aqygnziho > bsyrmrqsrq else iisukudgvu
line 1843: self.aqygnziho = 3
line 1983: self.aqygnziho -= 1
line 1984: if self.aqygnziho == 0:


In [30]:
for i, line in enumerate(lines[1978:2015], start=1979):
    print(f"line {i}: {line.rstrip()}")

line 1979:             self.next_level()
line 1980:             self.complete_action()
line 1981:             return
line 1982:         if bkuguqrpvq:
line 1983:             self.aqygnziho -= 1
line 1984:             if self.aqygnziho == 0:
line 1985:                 self.lose()
line 1986:                 self.complete_action()
line 1987:                 return
line 1988:             self.aqdxgoyvu.set_visible(True)
line 1989:             self.aqdxgoyvu.set_scale(64)
line 1990:             self.aqdxgoyvu.set_position(0, 0)
line 1991:             self.htkmubhry.set_visible(False)
line 1992:             self.ebfuxzbvn = xvzyzqolqz
line 1993:             self.lvrnuajbl = [False] * len(self.plrpelhym)
line 1994:             self.gudziatsk.set_position(self.ltwrkifkx, self.zyoimjaei)
line 1995:             self.qetwzqzzik()
line 1996:             for iybkldaxol in self.ofoahudlo:
line 1997:                 self.current_level.add_sprite(iybkldaxol)
line 1998:             for spssohsnbq in se

In [31]:
matches = [(i+1, line.strip()) for i, line in enumerate(lines) if "qetwzqzzik" in line]
for lineno, text in matches[:10]:
    print(f"line {lineno}: {text}")

line 1839: self.qetwzqzzik()
line 1995: self.qetwzqzzik()
line 2016: def qetwzqzzik(self) -> None:


In [32]:
for i, line in enumerate(lines[2015:2040], start=2016):
    print(f"line {i}: {line.rstrip()}")

line 2016:     def qetwzqzzik(self) -> None:
line 2017:         self.cklxociuu = self.dhksvilbb.index(self.current_level.get_data("StartRotation"))
line 2018:         self.hiaauhahz = self.tnkekoeuk.index(self.current_level.get_data("StartColor"))
line 2019:         self.fwckfzsyc = self.current_level.get_data("StartShape")
line 2020:         self.htkmubhry.pixels = self.ijessuuig[self.fwckfzsyc].pixels.copy()
line 2021:         self.htkmubhry.color_remap(epeqflmtfc, self.tnkekoeuk[self.hiaauhahz])
line 2022:         self.htkmubhry.set_rotation(self.dhksvilbb[self.cklxociuu])
line 2023: 
line 2024:     def vqfjzzkhid(self) -> bool:
line 2025:         if self.level_index > 0:
line 2026:             return False
line 2027:         uhftkonsvn = False
line 2028:         for wyhfvzxukh, spssohsnbq in enumerate(self.plrpelhym):
line 2029:             pidqulszaw = self.current_level.get_sprite_at(spssohsnbq.x - 1, spssohsnbq.y - 1, "hoswmpiqkw")
line 2030:             if self.bejndxqqzf(wyhfv